# Customer Churn Prediction — ML Model Comparison
Compares Logistic Regression, Random Forest, and XGBoost on a synthetic churn dataset.

In [ ]:
# Run this cell first to install all required packages
# !pip install xgboost lightgbm optuna shap scikit-learn pandas numpy matplotlib seaborn joblib streamlit

In [ ]:
import numpy as npy
import pandas as pds
import matplotlib.pyplot as mpl
import matplotlib.patches as mpatches
import seaborn as sbn
import joblib
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score,
    confusion_matrix, roc_curve
)
from xgboost import XGBClassifier

SEED = 42
npy.random.seed(SEED)
print('Libraries loaded.')

## 1. Synthetic Churn Dataset

In [ ]:
FEATURE_NAMES = [
    'tenure_months', 'monthly_charges', 'total_charges',
    'num_products', 'support_calls', 'payment_delay_days',
    'contract_length', 'online_security', 'tech_support',
    'streaming_tv', 'age', 'satisfaction_score',
    'data_usage_gb', 'late_payments', 'promo_discount'
]

X_raw, y = make_classification(
    n_samples=5000, n_features=15, n_informative=10,
    n_redundant=3, n_clusters_per_class=2,
    weights=[0.75, 0.25], flip_y=0.02,
    random_state=SEED
)

df = pds.DataFrame(X_raw, columns=FEATURE_NAMES)
df['churn'] = y

print(f'Dataset shape : {df.shape}')
print(f'Churn rate    : {y.mean():.1%}')
df.head()

## 2. Train / Test Split & Scaling

In [ ]:
X = df.drop('churn', axis=1)
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} rows  |  Test: {X_test.shape[0]} rows')

## 3. Model Training

### 3a. Logistic Regression (baseline)

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=SEED)
lr.fit(X_train_sc, y_train)
print('Logistic Regression trained.')

### 3b. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
print('Random Forest trained.')

### 3c. XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=SEED,
    verbosity=0
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
print('XGBoost trained.')

## 4. Evaluation

In [ ]:
def evaluate(name, model, X_te, y_te):
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    return {
        'Model'    : name,
        'Accuracy' : accuracy_score(y_te, y_pred),
        'ROC-AUC'  : roc_auc_score(y_te, y_proba),
        'F1'       : f1_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred),
        'Recall'   : recall_score(y_te, y_pred),
    }

results = pds.DataFrame([
    evaluate('Logistic Regression', lr,  X_test_sc, y_test),
    evaluate('Random Forest',       rf,  X_test,    y_test),
    evaluate('XGBoost',             xgb, X_test,    y_test),
])

results.set_index('Model', inplace=True)
results.round(4)

## 5. Model Comparison Chart

In [ ]:
metrics   = ['Accuracy', 'ROC-AUC', 'F1', 'Precision', 'Recall']
models    = results.index.tolist()
colors    = ['#4C72B0', '#55A868', '#C44E52']
x         = npy.arange(len(metrics))
bar_width = 0.22

fig, axes = mpl.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Customer Churn — Model Comparison', fontsize=15, fontweight='bold', y=1.01)

# ── Left: grouped bar chart ──────────────────────────────────────────────────
ax = axes[0]
for i, (model, color) in enumerate(zip(models, colors)):
    vals = results.loc[model, metrics].values
    bars = ax.bar(x + i * bar_width, vals, bar_width, label=model,
                  color=color, alpha=0.87, edgecolor='white', linewidth=0.6)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7.5, rotation=90)

ax.set_xticks(x + bar_width)
ax.set_xticklabels(metrics, fontsize=10)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Metric Comparison (Test Set)', fontsize=12)
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)

# ── Right: ROC curves ────────────────────────────────────────────────────────
ax2 = axes[1]
model_data = [
    ('Logistic Regression', lr,  X_test_sc),
    ('Random Forest',       rf,  X_test),
    ('XGBoost',             xgb, X_test),
]
for (name, model, Xte), color in zip(model_data, colors):
    proba = model.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax2.plot(fpr, tpr, color=color, lw=2, label=f'{name}  (AUC={auc:.3f})')

ax2.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
ax2.set_xlabel('False Positive Rate', fontsize=11)
ax2.set_ylabel('True Positive Rate', fontsize=11)
ax2.set_title('ROC Curves', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(linestyle='--', alpha=0.4)
ax2.spines[['top', 'right']].set_visible(False)

mpl.tight_layout()
mpl.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
mpl.show()
print('Chart saved to model_comparison.png')

## 6. XGBoost Feature Importance

In [ ]:
importance_df = pds.DataFrame({
    'Feature'   : FEATURE_NAMES,
    'Importance': xgb.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = mpl.subplots(figsize=(8, 6))
bars = ax.barh(importance_df['Feature'], importance_df['Importance'],
               color='#C44E52', alpha=0.85, edgecolor='white')
ax.set_xlabel('F-score (gain)', fontsize=11)
ax.set_title('XGBoost — Feature Importance', fontsize=13, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', linestyle='--', alpha=0.4)
mpl.tight_layout()
mpl.savefig('xgb_feature_importance.png', dpi=150, bbox_inches='tight')
mpl.show()
print('Feature importance chart saved.')

## 7. Confusion Matrices

In [ ]:
fig, axes = mpl.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices (Test Set)', fontsize=13, fontweight='bold')

for ax, (name, model, Xte) in zip(axes, model_data):
    cm = confusion_matrix(y_test, model.predict(Xte))
    sbn.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'],
                cbar=False, linewidths=0.5)
    ax.set_title(name, fontsize=11)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

mpl.tight_layout()
mpl.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
mpl.show()

## 8. Summary

| Model | Strengths | Weaknesses |
|---|---|---|
| Logistic Regression | Fast, interpretable, good baseline | Assumes linearity; lower recall on minority class |
| Random Forest | Robust to outliers, handles non-linearity | Slower inference, less tunable than XGBoost |
| **XGBoost** | **Best ROC-AUC & F1**, handles class imbalance via `scale_pos_weight` | Needs careful tuning; more hyperparameters |

**Recommendation:** Use **XGBoost** for production churn scoring. Tune `n_estimators`, `max_depth`, and `learning_rate` with cross-validation for further gains.

---
## 9. Hyperparameter Tuning with Optuna
Tunes both XGBoost and LightGBM using Bayesian optimisation (faster and smarter than GridSearchCV).

In [ ]:
import optuna
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
optuna.logging.set_verbosity(optuna.logging.WARNING)
print('Optuna and LightGBM loaded.')

### 9a. Tune XGBoost with Optuna

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def xgb_objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 600),
        'max_depth'        : trial.suggest_int('max_depth', 3, 10),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 10),
        'gamma'            : trial.suggest_float('gamma', 0, 5),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
        'scale_pos_weight' : (y_train == 0).sum() / (y_train == 1).sum(),
        'use_label_encoder': False,
        'eval_metric'      : 'logloss',
        'random_state'     : SEED,
        'verbosity'        : 0,
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=cv,
                             scoring='roc_auc', n_jobs=-1)
    return scores.mean()

xgb_study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print(f'\nBest XGBoost ROC-AUC (CV): {xgb_study.best_value:.4f}')
print('Best params:', xgb_study.best_params)

### 9b. Tune LightGBM with Optuna

In [ ]:
def lgb_objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 600),
        'max_depth'        : trial.suggest_int('max_depth', 3, 10),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 150),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
        'class_weight'     : 'balanced',
        'random_state'     : SEED,
        'verbosity'        : -1,
    }
    model = lgb.LGBMClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=cv,
                             scoring='roc_auc', n_jobs=-1)
    return scores.mean()

lgb_study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
lgb_study.optimize(lgb_objective, n_trials=50, show_progress_bar=True)

print(f'\nBest LightGBM ROC-AUC (CV): {lgb_study.best_value:.4f}')
print('Best params:', lgb_study.best_params)

### 9c. Train Tuned Models & Compare All Five

In [ ]:
xgb_tuned = XGBClassifier(
    **xgb_study.best_params,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=SEED,
    verbosity=0
)
xgb_tuned.fit(X_train, y_train)

lgb_tuned = lgb.LGBMClassifier(
    **lgb_study.best_params,
    class_weight='balanced',
    random_state=SEED,
    verbosity=-1
)
lgb_tuned.fit(X_train, y_train)

all_results = pds.DataFrame([
    evaluate('Logistic Regression',  lr,        X_test_sc, y_test),
    evaluate('Random Forest',        rf,        X_test,    y_test),
    evaluate('XGBoost (baseline)',   xgb,       X_test,    y_test),
    evaluate('XGBoost (tuned)',      xgb_tuned, X_test,    y_test),
    evaluate('LightGBM (tuned)',     lgb_tuned, X_test,    y_test),
])
all_results.set_index('Model', inplace=True)
all_results.round(4)

In [ ]:
metrics5  = ['Accuracy', 'ROC-AUC', 'F1', 'Precision', 'Recall']
model_names5 = all_results.index.tolist()
colors5   = ['#4C72B0', '#55A868', '#C44E52', '#E07B39', '#8172B2']
x5        = npy.arange(len(metrics5))
bw5       = 0.15

fig, ax = mpl.subplots(figsize=(14, 6))
for i, (name, color) in enumerate(zip(model_names5, colors5)):
    vals = all_results.loc[name, metrics5].values
    bars = ax.bar(x5 + i * bw5, vals, bw5, label=name,
                  color=color, alpha=0.87, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.004,
                f'{v:.3f}', ha='center', va='bottom', fontsize=6.5, rotation=90)

ax.set_xticks(x5 + bw5 * 2)
ax.set_xticklabels(metrics5, fontsize=11)
ax.set_ylim(0, 1.22)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('All Models — Tuned vs Baseline Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=8.5)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)
mpl.tight_layout()
mpl.savefig('tuned_model_comparison.png', dpi=150, bbox_inches='tight')
mpl.show()

---
## 10. SHAP Explainability
Waterfall, Beeswarm, and Force plots using the best tuned model.

In [ ]:
import shap
shap.initjs()

best_model  = xgb_tuned  # swap to lgb_tuned if LightGBM scored higher
explainer   = shap.TreeExplainer(best_model)
shap_values = explainer(X_test)

print(f'SHAP values shape: {shap_values.values.shape}')
print('Explainer ready.')

### 10a. Beeswarm Plot — Global Feature Impact

In [ ]:
mpl.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
mpl.title('SHAP Beeswarm — Global Feature Impact (XGBoost Tuned)',
          fontsize=12, fontweight='bold', pad=12)
mpl.tight_layout()
mpl.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
mpl.show()
print('Beeswarm saved.')

### 10b. Waterfall Plot — Single Prediction Explanation

In [ ]:
# Pick the first churner in the test set for explanation
churner_idx = npy.where(y_test.values == 1)[0][0]

mpl.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[churner_idx], max_display=15, show=False)
mpl.title(f'SHAP Waterfall — Customer #{churner_idx} (Predicted Churn)',
          fontsize=12, fontweight='bold', pad=12)
mpl.tight_layout()
mpl.savefig('shap_waterfall.png', dpi=150, bbox_inches='tight')
mpl.show()
print('Waterfall saved.')

### 10c. Force Plot — Interactive Single Prediction

In [ ]:
# Interactive force plot (renders inline in Jupyter)
shap.plots.force(
    shap_values[churner_idx],
    matplotlib=False   # set True for a static PNG instead
)

### 10d. SHAP Bar Plot — Mean Absolute Impact (Summary)

In [ ]:
mpl.figure(figsize=(9, 6))
shap.plots.bar(shap_values, max_display=15, show=False)
mpl.title('SHAP Bar — Mean |SHAP| per Feature', fontsize=12, fontweight='bold', pad=12)
mpl.tight_layout()
mpl.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
mpl.show()
print('SHAP bar plot saved.')

---
## 11. Updated Summary

| Model | ROC-AUC | Notes |
|---|---|---|
| Logistic Regression | baseline | Fast, interpretable |
| Random Forest | mid | Robust, no tuning |
| XGBoost (baseline) | good | Default hyperparams |
| **XGBoost (tuned)** | **best or near-best** | Optuna 50-trial search |
| **LightGBM (tuned)** | **best or near-best** | Faster training, comparable AUC |

**SHAP insights:** Features with the highest mean |SHAP| value (beeswarm/bar) are the true drivers of churn — use these to build targeted retention strategies.

**Install all dependencies:**
```bash
pip install xgboost lightgbm optuna shap scikit-learn pandas numpy matplotlib seaborn
```

---
## 12. Save Models & Artifacts for Streamlit
Run this cell **after** completing all sections above. It saves everything the Streamlit app needs so it doesn't retrain on every launch.

In [ ]:
import os
os.makedirs('models', exist_ok=True)

joblib.dump(xgb_tuned,    'models/xgb_tuned.pkl')
joblib.dump(lgb_tuned,    'models/lgb_tuned.pkl')
joblib.dump(scaler,       'models/scaler.pkl')
joblib.dump(FEATURE_NAMES,'models/feature_names.pkl')

X_test.to_csv('models/X_test.csv', index=False)
y_test.to_csv('models/y_test.csv', index=False)

print('Saved to models/:')
print('  xgb_tuned.pkl')
print('  lgb_tuned.pkl')
print('  scaler.pkl')
print('  X_test.csv')
print('  y_test.csv')
print('  feature_names.pkl')
print('\nYou are ready to run:  streamlit run churn_app.py')